# ラベルの説明文との類似度でゼロショット分類する

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/12_classification/zero_shot_embedding.ipynb)

NLP クックブック 12-15 ／ テキスト分類

ゼロショット分類を高速に行いたい。テキストと「ラベルの説明文」を同じ空間に埋め込み、最も近いラベルを選ぶ。

In [1]:
import numpy as np
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("intfloat/multilingual-e5-small")

label_desc = {
    "スポーツ": "野球やサッカーなどスポーツの試合や選手に関する話題",
    "経済": "金融、株価、企業業績、景気など経済に関する話題",
    "テクノロジー": "AI、ソフトウェア、スマートフォンなど技術に関する話題",
    "料理": "レシピ、食材、調理方法など料理に関する話題",
    "政治": "選挙、国会、政府、外交など政治に関する話題",
}
label_emb = model.encode(["query: " + d for d in label_desc.values()], normalize_embeddings=True)

texts = ["大谷翔平が今季 50 本目のホームランを放った。", "日銀は政策金利を据え置くことを決定した。",
         "新しい言語モデルは従来より少ない計算量で高い性能を達成した。", "玉ねぎを飴色になるまで炒めてから、カレールーを加えます。",
         "首相は来月の衆議院解散を表明した。"]
sims = model.encode(["query: " + t for t in texts], normalize_embeddings=True) @ label_emb.T
for text, row in zip(texts, sims):
    order = np.argsort(-row)
    names = list(label_desc)
    print(f"{names[order[0]]}({row[order[0]]:.2f}) / {names[order[1]]}({row[order[1]]:.2f})  {text}")

スポーツ(0.83) / 政治(0.82)  大谷翔平が今季 50 本目のホームランを放った。
経済(0.84) / 政治(0.83)  日銀は政策金利を据え置くことを決定した。
テクノロジー(0.83) / 経済(0.81)  新しい言語モデルは従来より少ない計算量で高い性能を達成した。
料理(0.85) / 政治(0.80)  玉ねぎを飴色になるまで炒めてから、カレールーを加えます。
政治(0.87) / 経済(0.84)  首相は来月の衆議院解散を表明した。


- ラベル名だけでなく、少し長い説明文や典型例を埋め込むと精度が上がる。各ラベルに数件の例文があるなら、その平均ベクトル（セントロイド）を使う。
- 1 位と 2 位のスコア差が小さいものは「分類不能」として人手に回す、という運用がしやすい。